In [1]:
import torch
import torchrl.envs as torch_envs
import gymnasium as gym
from spaceship_env import SpaceshipEnv
from ppo import PPO, PPOManager
from torchrl.record import VideoRecorder
from torchrl.record.loggers.csv import CSVLogger
device = torch.device("cuda")

/home/damian/miniconda3/envs/control/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
def make_norm_transforms(env: gym.Env):
    transforms = []
    for key, space in env.observation_space.items():
        if key in ["position", "target", "velocity", "rotation"]:
            transforms.append(torch_envs.transforms.ObservationNorm(loc=space.low, scale=1 / (space.high-space.low), in_keys=key, out_keys=key, standard_normal=False))
    return torch_envs.transforms.Compose(*transforms)

gym.register('Spaceship_Target', entry_point="spaceship_env:SpaceshipEnv")

env = torch_envs.GymEnv('Spaceship_Target', device=device)
print(env.observation_spec.keys())
env = torch_envs.transforms.TransformedEnv(base_env=env, 
                                             transform=torch_envs.Compose([
                                                 make_norm_transforms(env),
                                                 torch_envs.transforms.CatTensors(["position", "target", "velocity", "rotation"], "observation")
                                                 ]))

logged_env = torch_envs.GymEnv('Spaceship_Target', device=torch.device('cuda'), return_pixels=True)
logged_env = torch_envs.transforms.TransformedEnv(base_env=logged_env, 
                                             transform=torch_envs.Compose([
                                                 make_norm_transforms(env),
                                                 torch_envs.transforms.CatTensors(["position", "target", "velocity", "rotation"], "observation")
                                                 ]))

logger = CSVLogger(exp_name="Spaceship_Target", log_dir="target1_videos", video_format="mp4")
logged_env = torch_envs.transforms.TransformedEnv(logged_env, VideoRecorder(logger, tag="run_video", in_keys=['pixels'])) 

print(logged_env.observation_spec.keys())

print(env.rollout(3)['observation'])

_CompositeSpecKeysView(keys=['position', 'rotation', 'step_count', 'target', 'velocity'])
_CompositeSpecKeysView(keys=['step_count', 'pixels', 'observation'])
tensor([[ 7.1429e-02,  1.0000e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  5.1571e-01,  9.4200e-01,  0.0000e+00,  0.0000e+00],
        [ 7.1429e-02,  9.9840e-02,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  5.1571e-01,  9.4200e-01,  0.0000e+00, -8.0000e-04],
        [ 7.1429e-02,  9.9520e-02,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  5.1571e-01,  9.4200e-01,  0.0000e+00, -1.6000e-03]],
       device='cuda:0')


/home/damian/miniconda3/envs/control/lib/python3.12/site-packages/torchrl/envs/transforms/transforms.py:829: FutureWarning: The default behavior of TransformedEnv will change in version 0.9. Nested TransformedEnvs will no longer be automatically unwrapped by default. To prepare for this change, use set_auto_unwrap_transformed_env(val: bool) as a decorator or context manager, or set the environment variable AUTO_UNWRAP_TRANSFORMED_ENV to 'False'.
  instance: EnvBase = super(_EnvPostInit, self).__call__(*args, **kwargs)


In [3]:
type(env.action_spec.shape[-1])
env.observation_spec['observation'].shape[-1]

10

In [4]:
ppo = PPO(env.action_spec, env.observation_spec)
manager = PPOManager(ppo, env, logged_env)

In [5]:
manager.train(frames_per_batch=5000, total_frames=100_000_000, sub_batch_size=256, num_epochs=10)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x10 and 3x128)